# 05 - Evaluation Results

This notebook compares three systems:

1. TF-IDF Baseline  
2. Qwen LoRA  
3. RAG + Qwen LoRA  

Main metrics:
- ROUGE-1
- ROUGE-2
- ROUGE-L
- BLEU
- TF-IDF Answer Similarity
- Answer Length
- Compression Ratio

Retrieval metrics are only used for retrieval-based systems:
- TF-IDF Baseline
- RAG + Qwen LoRA

In [34]:
import json
from pathlib import Path

import pandas as pd
import numpy as np

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [35]:
BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

RESULTS_DIR = BASE_DIR / "results"

print("Base directory:", BASE_DIR)
print("Results directory:", RESULTS_DIR)
print("Results folder exists:", RESULTS_DIR.exists())

Base directory: d:\CardioBot_NLP_Final
Results directory: d:\CardioBot_NLP_Final\results
Results folder exists: True


In [36]:
required_files = {
    "baseline": RESULTS_DIR / "baseline_tfidf_answers.json",
    "lora": RESULTS_DIR / "finetuned_qwen_lora_answers.json",
    "rag": RESULTS_DIR / "rag_answers.json",
    "baseline_summary": RESULTS_DIR / "baseline_tfidf_summary.json",
    "rag_summary": RESULTS_DIR / "rag_summary.json"
}

for name, path in required_files.items():
    print(f"{name}: {path.exists()} -> {path.name}")

baseline: True -> baseline_tfidf_answers.json
lora: True -> finetuned_qwen_lora_answers.json
rag: True -> rag_answers.json
baseline_summary: True -> baseline_tfidf_summary.json
rag_summary: True -> rag_summary.json


In [37]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


baseline_results = load_json(required_files["baseline"])
lora_results = load_json(required_files["lora"])
rag_results = load_json(required_files["rag"])

baseline_summary = load_json(required_files["baseline_summary"])
rag_summary = load_json(required_files["rag_summary"])

print("Baseline results:", len(baseline_results))
print("LoRA results:", len(lora_results))
print("RAG results:", len(rag_results))

print("\nBaseline sample keys:", baseline_results[0].keys())
print("LoRA sample keys:", lora_results[0].keys())
print("RAG sample keys:", rag_results[0].keys())

Baseline results: 34
LoRA results: 34
RAG results: 34

Baseline sample keys: dict_keys(['id', 'topic', 'source', 'question', 'reference_answer', 'baseline_answer', 'top_source', 'top_score', 'retrieved_contexts'])
LoRA sample keys: dict_keys(['id', 'topic', 'source', 'question', 'reference_answer', 'finetuned_answer'])
RAG sample keys: dict_keys(['id', 'topic', 'source', 'question', 'reference_answer', 'rag_answer', 'top_source', 'top_score', 'retrieved_contexts'])


In [38]:
baseline_df = pd.DataFrame(baseline_results)
lora_df = pd.DataFrame(lora_results)
rag_df = pd.DataFrame(rag_results)

baseline_eval = baseline_df[[
    "id", "topic", "source", "question", "reference_answer", "baseline_answer"
]].copy()
baseline_eval["model"] = "TF-IDF Baseline"
baseline_eval = baseline_eval.rename(columns={"baseline_answer": "prediction"})

lora_eval = lora_df[[
    "id", "topic", "source", "question", "reference_answer", "finetuned_answer"
]].copy()
lora_eval["model"] = "Qwen LoRA"
lora_eval = lora_eval.rename(columns={"finetuned_answer": "prediction"})

rag_eval = rag_df[[
    "id", "topic", "source", "question", "reference_answer", "rag_answer"
]].copy()
rag_eval["model"] = "RAG + Qwen LoRA"
rag_eval = rag_eval.rename(columns={"rag_answer": "prediction"})

all_eval_df = pd.concat(
    [baseline_eval, lora_eval, rag_eval],
    ignore_index=True
)

all_eval_df["reference_answer"] = all_eval_df["reference_answer"].astype(str).fillna("")
all_eval_df["prediction"] = all_eval_df["prediction"].astype(str).fillna("")

print("Combined rows:", len(all_eval_df))
print(all_eval_df["model"].value_counts())

all_eval_df.head()

Combined rows: 102
model
TF-IDF Baseline    34
Qwen LoRA          34
RAG + Qwen LoRA    34
Name: count, dtype: int64


,id,topic,source,question,reference_answer,prediction,model
0,qa_047,Cardiomyopathy,Cardiomyopathy.txt,What is dilated cardiomyopathy?,Dilated cardiomyopathy is a type of cardiomyop...,Dilated cardiomyopathy. In this type of cardio...,TF-IDF Baseline
1,qa_057,Heart Valve Disease,Heart Valve Disease.txt,What is valve regurgitation?,Valve regurgitation happens when valve flaps d...,Heart valve disease. An echocardiogram can sho...,TF-IDF Baseline
2,qa_099,Blood Flow,Blood_Flow.txt,What are the main functions of blood flow?,Blood flow delivers oxygen and nutrients to or...,"Blood flow has two main functions, which are d...",TF-IDF Baseline
3,qa_135,Heart Failure,heart_failure.txt,What are the ACC/AHA stages of heart failure?,The ACC/AHA stages of heart failure are Stage ...,"Heart failure, also known as congestive heart ...",TF-IDF Baseline
4,qa_070,Stroke,Stroke.txt,How is hemorrhagic stroke treated?,Hemorrhagic stroke treatment focuses on contro...,"If you have a hemorrhagic stroke, your provide...",TF-IDF Baseline


In [39]:
rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

smooth = SmoothingFunction().method1


def compute_rouge(reference, prediction):
    score = rouge.score(reference, prediction)
    return {
        "rouge_1": score["rouge1"].fmeasure,
        "rouge_2": score["rouge2"].fmeasure,
        "rouge_l": score["rougeL"].fmeasure
    }


def compute_bleu(reference, prediction):
    reference_tokens = reference.lower().split()
    prediction_tokens = prediction.lower().split()
    
    if len(prediction_tokens) == 0:
        return 0.0
    
    return sentence_bleu(
        [reference_tokens],
        prediction_tokens,
        smoothing_function=smooth
    )


def compute_tfidf_similarity(reference, prediction):
    if not reference.strip() or not prediction.strip():
        return 0.0
    
    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform([reference, prediction])
    
    return float(cosine_similarity(matrix[0], matrix[1])[0][0])


def word_count(text):
    return len(str(text).split())

In [40]:
rouge_scores = all_eval_df.apply(
    lambda row: compute_rouge(row["reference_answer"], row["prediction"]),
    axis=1
)

rouge_df = pd.DataFrame(list(rouge_scores))

all_eval_df = pd.concat(
    [all_eval_df.reset_index(drop=True), rouge_df.reset_index(drop=True)],
    axis=1
)

all_eval_df["bleu"] = all_eval_df.apply(
    lambda row: compute_bleu(row["reference_answer"], row["prediction"]),
    axis=1
)

all_eval_df["answer_similarity"] = all_eval_df.apply(
    lambda row: compute_tfidf_similarity(row["reference_answer"], row["prediction"]),
    axis=1
)

all_eval_df["reference_word_count"] = all_eval_df["reference_answer"].apply(word_count)
all_eval_df["prediction_word_count"] = all_eval_df["prediction"].apply(word_count)

all_eval_df["compression_ratio"] = (
    all_eval_df["prediction_word_count"] / all_eval_df["reference_word_count"]
)

metric_cols = [
    "rouge_1", "rouge_2", "rouge_l",
    "bleu", "answer_similarity",
    "reference_word_count", "prediction_word_count",
    "compression_ratio"
]

for col in metric_cols:
    all_eval_df[col] = pd.to_numeric(all_eval_df[col], errors="coerce")

all_eval_df[[
    "model", "question", "rouge_1", "rouge_2", "rouge_l",
    "bleu", "answer_similarity", "prediction_word_count", "compression_ratio"
]].head()

,model,question,rouge_1,rouge_2,rouge_l,bleu,answer_similarity,prediction_word_count,compression_ratio
0,TF-IDF Baseline,What is dilated cardiomyopathy?,0.577778,0.318182,0.488889,0.038071,0.454750,51,1.378378
1,TF-IDF Baseline,What is valve regurgitation?,0.212121,0.031250,0.151515,0.010396,0.306647,47,2.473684
2,TF-IDF Baseline,What are the main functions of blood flow?,0.455285,0.347107,0.406504,0.153324,0.402717,89,2.617647
3,TF-IDF Baseline,What are the ACC/AHA stages of heart failure?,0.238806,0.045455,0.149254,0.018238,0.199958,80,1.509434
4,TF-IDF Baseline,How is hemorrhagic stroke treated?,0.452830,0.211538,0.320755,0.033103,0.425744,71,2.218750


In [41]:
print("Rows per model:")
print(all_eval_df["model"].value_counts())

print("\nMissing predictions:")
print(all_eval_df.groupby("model")["prediction"].apply(lambda x: x.isna().sum()))

print("\nEmpty predictions:")
print(all_eval_df.groupby("model")["prediction"].apply(lambda x: (x.str.strip() == "").sum()))

print("\nMetric null count:")
print(
    all_eval_df.groupby("model")[
        ["rouge_1", "rouge_2", "rouge_l", "bleu", "answer_similarity"]
    ].apply(lambda x: x.isna().sum())
)

Rows per model:
model
TF-IDF Baseline    34
Qwen LoRA          34
RAG + Qwen LoRA    34
Name: count, dtype: int64

Missing predictions:
model
Qwen LoRA          0
RAG + Qwen LoRA    0
TF-IDF Baseline    0
Name: prediction, dtype: int64

Empty predictions:
model
Qwen LoRA          0
RAG + Qwen LoRA    0
TF-IDF Baseline    0
Name: prediction, dtype: int64

Metric null count:
                 rouge_1  rouge_2  rouge_l  bleu  answer_similarity
model                                                              
Qwen LoRA              0        0        0     0                  0
RAG + Qwen LoRA        0        0        0     0                  0
TF-IDF Baseline        0        0        0     0                  0


In [42]:
summary_df = (
    all_eval_df
    .groupby("model", dropna=False)
    .agg(
        test_size=("id", "count"),
        avg_rouge_1=("rouge_1", "mean"),
        avg_rouge_2=("rouge_2", "mean"),
        avg_rouge_l=("rouge_l", "mean"),
        avg_bleu=("bleu", "mean"),
        avg_answer_similarity=("answer_similarity", "mean"),
        avg_reference_length=("reference_word_count", "mean"),
        avg_prediction_length=("prediction_word_count", "mean"),
        avg_compression_ratio=("compression_ratio", "mean")
    )
    .reset_index()
)

numeric_cols = [
    "avg_rouge_1",
    "avg_rouge_2",
    "avg_rouge_l",
    "avg_bleu",
    "avg_answer_similarity",
    "avg_reference_length",
    "avg_prediction_length",
    "avg_compression_ratio"
]

summary_df[numeric_cols] = summary_df[numeric_cols].round(4)

summary_df

,model,test_size,avg_rouge_1,avg_rouge_2,avg_rouge_l,avg_bleu,avg_answer_similarity,avg_reference_length,avg_prediction_length,avg_compression_ratio
0,Qwen LoRA,34,0.4621,0.2320,0.3793,0.1302,0.3624,34.0588,33.0588,1.0225
1,RAG + Qwen LoRA,34,0.5318,0.3008,0.4517,0.1648,0.4398,34.0588,30.0294,0.8944
2,TF-IDF Baseline,34,0.2972,0.1193,0.2180,0.0408,0.2897,34.0588,71.8824,2.2307


In [43]:
retrieval_metrics = pd.DataFrame([
    {
        "model": "TF-IDF Baseline",
        "source_match_accuracy": baseline_summary.get("source_match_accuracy"),
        "top3_source_match_accuracy": baseline_summary.get("top3_source_match_accuracy"),
        "average_top_score": baseline_summary.get("average_top_score")
    },
    {
        "model": "RAG + Qwen LoRA",
        "source_match_accuracy": rag_summary.get("source_match_accuracy"),
        "top3_source_match_accuracy": rag_summary.get("top3_source_match_accuracy"),
        "average_top_score": rag_summary.get("average_top_score")
    },
    {
        "model": "Qwen LoRA",
        "source_match_accuracy": np.nan,
        "top3_source_match_accuracy": np.nan,
        "average_top_score": np.nan
    }
])

retrieval_metrics

,model,source_match_accuracy,top3_source_match_accuracy,average_top_score
0,TF-IDF Baseline,0.6176,0.8529,0.2919
1,RAG + Qwen LoRA,0.6176,0.8529,0.2919
2,Qwen LoRA,NaN,NaN,NaN


In [44]:
final_comparison = summary_df.merge(
    retrieval_metrics,
    on="model",
    how="left"
)

final_comparison = final_comparison[
    [
        "model",
        "test_size",
        "avg_rouge_1",
        "avg_rouge_2",
        "avg_rouge_l",
        "avg_bleu",
        "avg_answer_similarity",
        "avg_reference_length",
        "avg_prediction_length",
        "avg_compression_ratio",
        "source_match_accuracy",
        "top3_source_match_accuracy",
        "average_top_score"
    ]
]

final_comparison

,model,test_size,avg_rouge_1,avg_rouge_2,avg_rouge_l,avg_bleu,avg_answer_similarity,avg_reference_length,avg_prediction_length,avg_compression_ratio,source_match_accuracy,top3_source_match_accuracy,average_top_score
0,Qwen LoRA,34,0.4621,0.2320,0.3793,0.1302,0.3624,34.0588,33.0588,1.0225,NaN,NaN,NaN
1,RAG + Qwen LoRA,34,0.5318,0.3008,0.4517,0.1648,0.4398,34.0588,30.0294,0.8944,0.6176,0.8529,0.2919
2,TF-IDF Baseline,34,0.2972,0.1193,0.2180,0.0408,0.2897,34.0588,71.8824,2.2307,0.6176,0.8529,0.2919


In [45]:
final_comparison.to_csv(
    RESULTS_DIR / "final_model_comparison.csv",
    index=False
)

all_eval_df.to_csv(
    RESULTS_DIR / "all_model_predictions_with_metrics.csv",
    index=False
)

with open(RESULTS_DIR / "final_model_comparison.json", "w", encoding="utf-8") as f:
    json.dump(final_comparison.to_dict("records"), f, indent=2)

print("Saved:")
print(RESULTS_DIR / "final_model_comparison.csv")
print(RESULTS_DIR / "all_model_predictions_with_metrics.csv")
print(RESULTS_DIR / "final_model_comparison.json")

Saved:
d:\CardioBot_NLP_Final\results\final_model_comparison.csv
d:\CardioBot_NLP_Final\results\all_model_predictions_with_metrics.csv
d:\CardioBot_NLP_Final\results\final_model_comparison.json


In [46]:
best_rouge_model = final_comparison.sort_values(
    "avg_rouge_l", ascending=False
).iloc[0]["model"]

best_similarity_model = final_comparison.sort_values(
    "avg_answer_similarity", ascending=False
).iloc[0]["model"]

print("Evaluation Interpretation:")
print(f"- Best model based on ROUGE-L: {best_rouge_model}")
print(f"- Best model based on answer similarity: {best_similarity_model}")


Evaluation Interpretation:
- Best model based on ROUGE-L: RAG + Qwen LoRA
- Best model based on answer similarity: RAG + Qwen LoRA


TF-IDF baseline is simple and interpretable, but its answers are extractive and may include less focused context.

Qwen LoRA produces generated answers based on fine-tuning, but it does not retrieve document context during inference.

RAG + Qwen LoRA combines document retrieval and fine-tuned generation, making the answers more grounded in the cardiovascular document collection."

Retrieval metrics are not applicable to Qwen LoRA-only because it does not retrieve external context.